In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

print('Libraries loaded.')

Libraries loaded.



## Feature Engineering


In [2]:
df= pd.read_csv("../data/clean_results.csv")
df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'year', 'decade',
       'total_goals'],
      dtype='object')

### 1. `goal_difference_abs`

In [3]:
df['goal_difference'] = df['home_score'] - df['away_score']
df["goal_difference_abs"] = df["goal_difference"].abs()

### 2 `match_result`
`Home Team Win`, `Away Team Win`, or `Draw`  


In [4]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return "Home Team Win"
    elif row['home_score'] < row['away_score']:
        return "Away Team Win"
    else:
        return "Draw"

df["match_result"] = df.apply(get_result,axis=1)

### 3. `home_win`
`1` if Home Win, `0` otherwise  


In [5]:
df["home_win"] = (df["match_result"] == "Home Team Win").astype(int)

### 4. `home_points` and `away_points`
`Win` = 3, `Draw` = 1, `Loss` = 0 


In [6]:
df['home_points'] = df['match_result'].map({
    'Home Team Win': 3,
    'Draw': 1,
    'Away Team Win': 0
})

df['away_points'] = df['match_result'].map({
    'Home Team Win': 0,
    'Draw': 1,
    'Away Team Win': 3
})

### 5. `high_scoring_match`
`Yes` if `total_goals >= 4`, otherwise `No`  


In [7]:
df["high_scoring_match"] = df["total_goals"].apply(lambda x: "Yes" if x>=4 else "No")

### 6. `goal_margin_category`
based on `goal_difference_abs`:

 `Draw` = 0 goals difference |
 `Close Match` = 1 goal difference |
 `Comfortable Win` = 2–3 goals difference |
 `Dominant Win` = 4+ goals difference 


In [8]:
def goal_margin_cat(row):
    diff = row['goal_difference_abs']
    result = row['match_result']

    if result == 'Draw':
        return 'Draw'
    elif diff == 1:
        return 'Close Match'
    elif diff <= 3:
        return 'Comfortable Win'
    else:
        return 'Dominant Win'

df['goal_margin_category'] = df.apply(goal_margin_cat, axis=1)
    

### 7. `is_neutral`
`1` = neutral venue, `0` = home/away venue  


In [9]:
# Convert boolean neutral column to integer (1/0)
df['is_neutral'] = df['neutral'].astype(int)
df=df.drop("neutral",axis=1)

#### Summarize and save prepared dataset 

In [10]:
# List all columns with their data types
col_summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
})
col_summary


,Column,Data Type
0,date,object
1,home_team,object
2,away_team,object
3,home_score,float64
4,away_score,float64
5,tournament,object
6,city,object
7,country,object
8,year,int64
9,decade,int64


In [11]:
print(f'Final shape : {df.shape[0]:,} rows x {df.shape[1]} columns')

Final shape : 49,425 rows x 20 columns


In [12]:
df.to_csv("../data/matches_prepared.csv",index=False)